在本章中，我们将继续使用PyOpenCL的实践经验。我们将遵循我们在上一章讨论的CUDA项目示例，并尝试与ROCm的HIP进行跨平台合作。为了做到这一点，我们将使用hipify工具，我们在第4章，GPU编程的基础知识中学习到了这个工具。使用hipify，我们将尝试从.cu文件创建.cpp文件。通过实际的方法，我们将看到将CUDA代码转换为HIP代码的整个过程。

C编程爱好者将被鼓励使用HIP在程序代码中调用AMD和NVIDIA gpu，而Python编程爱好者将被鼓励使用PyOpenCL在程序代码中调用AMD和NVIDIA gpu。我们将首先了解ROCm HIP-C程序是如何工作的。OpenCL程序及其组件背后的基本概念将通过一些基本示例进行讨论，这些示例将帮助您向pyopenclcuda环境过渡。PyOpenCL将用一种实用的方法进行解释，并与OpenCL进行比较。我们将重点讨论PyOpenCL对OpenCL的适用性偏好，同样涉及Python语法的简单性和强大性。

本章包含以下主题：
- 了解ROMCM/C++如何与HIPIDENT、HIP和OpenCL一起工作
- 安装PyOpenCL for Python（AMD和NVIDIA）
- 在Python IDE上配置PyOpenCL
- PyOpenCL中的计算如何在Python上工作
- 比较PyOpenCL与HIP和OpenCL–重新审视复位角度
- 编写第一个PyOpenCL程序来计算通用解决方案
- 关于计算问题解决的有用练习

# 了解ROMCM/C++如何与HIPIDENT、HIP和OpenCL一起工作

在本节中，我们将学习如何将CUDA代码转换为跨平台的HIP代码，以及如何使用HIP编译器编译移植的代码。最后，我们将通过一个OpenCL实例与CUDA的文档进行比较，从而更容易地理解开放计算语言。

## 用hipify将CUDA代码转换成跨平台HIP代码

当我们开始了解AMD和NVIDIA GPU的ROCm时，还有什么比将本书中的第一个CUDA程序转换为ROCm HIP版本更实际的方法呢？按照以下步骤实现：

1. 确保在安装gpu的位置打开终端_乘法.cu为理解CUDA而创建的文件。
2. 假设您已经安装了hipify，如前面第4章“GPU编程基础”中所述，让我们使用以下命令将.cu程序移植到更熟悉的.cpp程序扩展中：

我们现在已经把CUDA搞定了！注意在转换的代码中将cuda前缀替换为hip。

3. 以下是HIPified程序（(gpu_multiply_ported_to_hip.cpp）

hipMalloc与cudaMalloc类似，它使用GPU设备上的变量分配内存：

请注意，仅为HIP手动修改了以下注释：

4. hipEventElapsedTime将计算时间通过elapsedTime存储在以下代码中：

5. 下一步是验证通过GPU进行的乘法是否正确：

在此之后，我们通过执行同一个例子来检查ROMCM/C++如何与HIP一起工作。

## 了解ROCM-C/C++与HIP的关系

现在我们已经有了从CUDA移植的现成代码，让我们来了解HIP是如何工作的。为了获得可执行输出，我们将用C++ 11标准在HIGPC编译程序时如下：

现在，我们使用以下命令执行程序：

### NVIDIA平台上的输出

使用HIP编译器创建可执行文件可获得以下输出，如前一节所示：

我们可以看到，在基于Maxwell架构的nvidiatitanxgpu上，具有HIP的multiply函数的执行时间是51.659714毫秒。

### AMD平台上的输出

现在，让我们在基于AMD Radeon VII GPU的全新Vega 20体系结构上编译并测试相同的代码！这个新系统还安装了ROCm，如前面第4章GPU编程基础所示。我们得到以下输出：

Radeon VII（报告为Vega 20）在31.587000毫秒内执行相同的代码。两个系统的配置如下所示：

<table border="1" style="border-collapse: collapse;width: 100%">
<tbody>
<tr>
<td class="CDPAlignCenter CDPAlign">
<p><strong>Hardware Configuration 1</strong></p>
</td>
<td class="CDPAlignCenter CDPAlign">
<p><strong>Hardware Configuration 2</strong></p>
</td>
</tr>
<tr>
<td class="CDPAlignCenter CDPAlign">
<p>Intel Core i7 4770K processor at 3.5 Ghz</p>
</td>
<td class="CDPAlignCenter CDPAlign">
<p>AMD Ryzen 7 2700X processor at 3.7 Ghz</p>
</td>
</tr>
<tr>
<td class="CDPAlignCenter CDPAlign">
<p>NVIDIA Titan X 12 GB DDR5 GPU (Maxwell architecture)</p>
</td>
<td class="CDPAlignCenter CDPAlign">
<p>AMD Radeon VII 16 GB HBM2 GPU (Vega 20 architecture)</p>
</td>
</tr>
<tr>
<td class="CDPAlignCenter CDPAlign">
<p>32 GB GSkill DDR3 RAM</p>
</td>
<td class="CDPAlignCenter CDPAlign">
<p>32 GB Corsair DDR4 RAM</p>
</td>
</tr>
<tr>
<td class="CDPAlignCenter CDPAlign">
<p>256 GB Adata SSD</p>
</td>
<td class="CDPAlignCenter CDPAlign">
<p>500 GB WD M2 SSD</p>
</td>
</tr>
<tr>
<td class="CDPAlignCenter CDPAlign">
<p>2 TB Western Digital HDD Green</p>
</td>
<td class="CDPAlignCenter CDPAlign">
<p>4 TB Western Digital HDD Red</p>
</td>
</tr>
</tbody>
</table>

在上一章中，我们在CUDA部分讨论了线程、块和网格的概念。现在让我们看看hipify工具是如何对我们以前的CUDA代码进行修改的，以便生成一个包含类似于CUDA的GPU代码的交叉兼容.cpp程序。

在第一行中，我们注意到包含了一个新的头文件`hip_runtime.h`：

因此，从GPU全局函数的索引开始，我们发现语法保持不变，与CUDA版本（threadIdx.x、blockIdx.x和blockDim.x）没有区别。

我们注意到的下一个变化是cudaMalloc变为hipMalloc。cudaMemcpy和cudaMemcpyHostToDevice已分别更改为hipMemcpy和hipMemcpyHostToDevice。

让我们借助一个对比表来看看我们新髋关节项目的所有变化：

<table border="1" style="border-collapse: collapse;width: 100%">
<tbody>
<tr>
<td class="CDPAlignCenter CDPAlign">
<p><strong>CUDA program</strong></p>
</td>
<td class="CDPAlignCenter CDPAlign">
<p><strong>HIPified program</strong></p>
</td>
</tr>
<tr>
<td>
<p><kbd>gpu_multiply.cu</kbd></p>
</td>
<td>
<p><kbd>gpu_multiply_ported_to_hip.cpp</kbd></p>
</td>
</tr>
<tr>
<td>
<p>N/A</p>
</td>
<td>
<p><kbd>#include &lt;hip/hip_runtime.h&gt;</kbd></p>
</td>
</tr>
<tr>
<td>
<p><kbd>cudaMalloc</kbd></p>
</td>
<td>
<p><kbd>hipMalloc</kbd></p>
</td>
</tr>
<tr>
<td>
<p><kbd>cudaMemcpy</kbd></p>
</td>
<td>
<p><kbd>hipMemcpy</kbd></p>
</td>
</tr>
<tr>
<td>
<p><kbd>cudaMemcpyHostToDevice</kbd></p>
</td>
<td>
<p><kbd>hipMemcpyHostToDevice</kbd></p>
</td>
</tr>
<tr>
<td>
<p><kbd>cudaMemcpyDeviceToHost</kbd></p>
</td>
<td>
<p><kbd>hipMemcpyDeviceToHost</kbd></p>
</td>
</tr>
<tr>
<td>
<p><kbd>cudaEvent_t</kbd></p>
</td>
<td>
<p><kbd>hipEvent_t</kbd></p>
</td>
</tr>
<tr>
<td>
<p><kbd>cudaEventCreate</kbd></p>
</td>
<td>
<p><kbd>hipEventCreate</kbd></p>
</td>
</tr>
<tr>
<td>
<p><kbd>cudaEventRecord</kbd></p>
</td>
<td>
<p><kbd>hipEventRecord</kbd></p>
</td>
</tr>
<tr>
<td>
<p><kbd>cudaEventSynchronize</kbd></p>
</td>
<td>
<p><kbd>hipEventSynchronize</kbd></p>
</td>
</tr>
<tr>
<td>
<p><kbd>cudaEventElapsedTime</kbd></p>
</td>
<td>
<p><kbd>hipEventElapsedTime</kbd></p>
</td>
</tr>
<tr>
<td>
<p><kbd>cudaDeviceProp</kbd></p>
</td>
<td>
<p><kbd>hipDeviceProp_t</kbd></p>
</td>
</tr>
<tr>
<td>
<p><kbd>cudaGetDeviceCount</kbd></p>
</td>
<td>
<p><kbd>hipGetDeviceCount</kbd></p>
</td>
</tr>
<tr>
<td>
<p><kbd>cudaGetDeviceProperties</kbd></p>
</td>
<td>
<p><kbd>hipGetDeviceProperties</kbd></p>
</td>
</tr>
<tr>
<td>
<p><kbd>cudaFree</kbd></p>
</td>
<td>
<p><kbd>hipFree</kbd></p>
</td>
</tr>
<tr>
<td>
<p><kbd>multiply &lt;&lt;&lt;&gt;&gt;&gt;</kbd> (kernel invocation)</p>
</td>
<td>
<p><kbd>hipLaunchKernelGGL(multiply)</kbd> (kernel invocation)</p>
</td>
</tr>
</tbody>
</table>

在CUDA中，我们的内核调用如下：

现在，在HIP计划中，我们发现最显著的变化如下：

hipLaunchKernelGGL启动GPU内核以在GPU上执行内核函数。我们可以注意到dim3整数向量数据类型用于指定线程和块的数量。

在HIP程序中，我们注意到hipMalloc专门为GPU-only变量分配内存。在这种情况下，像CUDA一样，我们必须指定将变量从CPU传输到GPU，并将计算结果再次带回CPU。为了从CPU或GPU访问变量，HIP还不支持使用统一内存分配（hipMallocManaged和hipMemPrefetchAsync）。

为了确认这一点，我们将再次使用hipify：

代码已转换，但cudaMallocManaged和cudaMemPrefetchAsync除外。请注意，代码中的注释部分将被忽略：

`cudaMalloc`分配在测试统一内存分配（UMA）的代码中被注释掉：

在这里，`cudaMallocManaged` for UMA是不变的，就像上一章一样，HIP目前不支持UMA：

`cudaMemcpy`分配被注释掉，因为UMA不需要它们：

转换后的CUDA代码中最明显的变化是GPU内核调用：

以下代码部分验证结果和计算时间：

hipFree只是cudaFree的HIP替代品，目的是在通过代码使用内存分配后最终清除内存分配：

编译过程中出现警告但没有错误，因此让我们尝试执行程序：

我们可以通过在前一章的一个简单的例子中了解CUAD-C/C++的工作原理来获得一个熟悉的输出。所以，这个程序部分使用了NVCC和HIP代码。希望将来能够支持hipMallocManaged和hipMemPrefetchAsync。

## 了解OpenCL的工作原理

现在让我们看看OpenCL是如何工作的。请注意，OpenCL独立于ROCm，但在安装ROCm时也会提供它。

OpenCL是一种跨平台语言，相对来说比CUDA更复杂，但学习起来并不困难。为了便于理解OpenCL，让我们通过下面的OpenCL程序将其语法与CUDA的语法进行比较。它计算的操作与我们在上一章中对CUDA所做的操作类似。

首先要注意的是，OpenCL内核代码将被写入一个扩展名为.cl的单独内核文件中。与CUDA.cu文件相反，OpenCL.cl文件只包含内核代码，这相当于我们在CUDA上编写的全局函数以及主代码。在OpenCL中，主代码以一种传统的格式编写为.c或.cpp文件，其中.cl文件被读取并用于OpenCL设备上的计算。在本例中，我们使用.cpp扩展名。

这一次，我们将限制为1500万个元素，并将结果存储在第三个数组中，而不是更新第二个数组。

一个优点是我们的内核代码总是保持分离。这样，我们可以分别组织和关注我们的通用代码语法和并行代码语法，这使得代码管理更加方便。

首先，让我们看看OpenCL内核文件(gpu_multiply_kernel.cl):

注意，这里使用的语法不是CUDA上的全局语法，而是定义函数的内核语法。

我们不使用`threadIdx.x + blockIdx.x * blockDim.x`作为索引，而是使用`get_global_id(0)`。

对于主程序，我们使用以下语法来对比CUDA。强烈建议通过比较OpenCL和CUDA来理解代码行之间的每个注释。

`main.cpp`包含通用C/C++代码，如下面的代码块所示：

这里，我们使用` gpu_multiply_kernel.cl`在我们的主要节目中：

现在，我们获取平台和设备ID：

clCreateBuffer和clEnqueueWriteBuffer分别类似于CUDA中使用的cudaMalloc和cudaMemcpyHostToDevice：

现在，我们创建并构建OpenCL程序和内核。我们还传递内核参数：

这里，我们为OpenCL内核分配工作项和组。为了在熟悉度方面更好地进行关联，请注意，在CUDA中，工作项分别对应于线程，组分别对应于块：

以下代码用于获取所有OpenCL设备：

现在显示设备（GPU）名称：

以下代码段将使脚本等待事件完成：

为了使用OpenCL评测记录计算时间，我们使用clGetEventProfilingInfo。请注意，与CUDA（以毫秒为单位计算时间）不同，OpenCL以纳秒为单位计算时间：

clEnqueueReadBuffer与CUDA上的cudaMemcpyDeviceToHost类似：

对于清除内存分配，clreleasemobject类似于CUDA上的cudaFree：

使用以下命令编译和执行此程序：

clEnqueueReadBuffer与CUDA上的cudaMemcpyDeviceToHost类似：

对于清除内存分配，clreleasemobject类似于CUDA上的cudaFree：

使用以下命令编译和执行此程序：

对于1500万个元素，存储在第三个数组上的相同乘法值都在OpenCL设备上以33.358656毫秒的时间计算出来。

在下一节中，我们将开始探索如何利用OpenCL和Python代码的简单性，考虑到OpenCL与CUDA相比的复杂语法，这一点现在更加重要。当我们开始使用PyOpenCL进行计算时，我们将探讨前面提到的功能和并行特性的范围，特别是在Python中。虽然PyCUDA允许您同时使用OpenCL和Python代码，但我们的重点将再次放在充分利用针对AMD和NVIDIA gpu的Python编程方法上。

从这一点开始，在介绍了PyOpenCL之后，我们将逐步过渡到只使用Python的编程环境。但是，在我们这样做之前，我们要将基于C/C++的三种计算语言总结为GPU计算，通过比较它们基于官方ROM文档的句法差异：即CUDA、HIP和OpenCL。这非常有助于概括我们到目前为止研究的三种计算语言：

<table border="1" style="border-collapse: collapse;width: 100%">
<tbody>
<tr>
<td>
<p><strong>Term</strong></p>
</td>
<td>
<p><strong>CUDA</strong></p>
</td>
<td>
<p><strong>HIP</strong></p>
</td>
<td>
<p><strong>OpenCL</strong></p>
</td>
</tr>
<tr>
<td>
<p><strong>Device</strong></p>
</td>
<td>
<p><kbd>int deviceId</kbd></p>
</td>
<td>
<p><kbd>int deviceId</kbd></p>
</td>
<td>
<p><kbd>cl_device</kbd></p>
</td>
</tr>
<tr>
<td>
<p><strong>Queue</strong></p>
</td>
<td>
<p><kbd>cudaStream_t</kbd></p>
</td>
<td>
<p><kbd>hipStream_t</kbd></p>
</td>
<td>
<p><kbd>cl_command_queue</kbd></p>
</td>
</tr>
<tr>
<td>
<p><strong>Event</strong></p>
</td>
<td>
<p><kbd>cudaEvent_t</kbd></p>
</td>
<td>
<p><kbd>hipEvent_t</kbd></p>
</td>
<td>
<p><kbd>cl_event</kbd></p>
</td>
</tr>
<tr>
<td>
<p><strong>Memory</strong></p>
</td>
<td>
<p><kbd>void *</kbd></p>
</td>
<td>
<p><kbd>void *</kbd></p>
</td>
<td>
<p><kbd>cl_mem</kbd></p>
</td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td></td>
<td>
<p>grid</p>
</td>
<td>
<p>grid</p>
</td>
<td>
<p>NDRange</p>
</td>
</tr>
<tr>
<td></td>
<td>
<p>block</p>
</td>
<td>
<p>block</p>
</td>
<td>
<p>work-group</p>
</td>
</tr>
<tr>
<td></td>
<td>
<p>thread</p>
</td>
<td>
<p>thread</p>
</td>
<td>
<p>work-item</p>
</td>
</tr>
<tr>
<td></td>
<td>
<p>warp</p>
</td>
<td>
<p>warp</p>
</td>
<td>
<p>sub-group</p>
</td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td>
<p><strong>Thread-index</strong></p>
</td>
<td>
<p><kbd>threadIdx.x</kbd></p>
</td>
<td>
<p><kbd>hipThreadIdx_x</kbd></p>
</td>
<td>
<p><kbd>get_local_id(0)</kbd></p>
</td>
</tr>
<tr>
<td>
<p><strong>Block-index</strong></p>
</td>
<td>
<p><kbd>blockIdx.x</kbd></p>
</td>
<td>
<p><kbd>hipBlockIdx_x</kbd></p>
</td>
<td>
<p><kbd>get_group_id(0)</kbd></p>
</td>
</tr>
<tr>
<td>
<p><strong>Block-dim</strong></p>
</td>
<td>
<p><kbd>blockDim.x</kbd></p>
</td>
<td>
<p><kbd>hipBlockDim_x</kbd></p>
</td>
<td>
<p><kbd>get_local_size(0)</kbd></p>
</td>
</tr>
<tr>
<td>
<p><strong>Grid-dim</strong></p>
</td>
<td>
<p><kbd>gridDim.x</kbd></p>
</td>
<td>
<p><kbd>hipGridDim_x</kbd></p>
</td>
<td>
<p><kbd>get_global_size(0)</kbd></p>
</td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td>
<p><strong>Device Kernel</strong></p>
</td>
<td>
<p><kbd>__global__</kbd></p>
</td>
<td>
<p><kbd>__global__</kbd></p>
</td>
<td>
<p><kbd>__kernel</kbd></p>
</td>
</tr>
<tr>
<td>
<p><strong>Device Function</strong></p>
</td>
<td>
<p><kbd>__device__</kbd></p>
</td>
<td>
<p><kbd>__device__</kbd></p>
</td>
<td>
<p>Implied in device compilation</p>
</td>
</tr>
<tr>
<td>
<p><strong>Host Function</strong></p>
</td>
<td>
<p><kbd>__host_ (default)</kbd></p>
</td>
<td>
<p><kbd>__host_ (default)</kbd></p>
</td>
<td>
<p>Implied in host compilation.</p>
</td>
</tr>
<tr>
<td>
<p><strong>Host + Device Function</strong></p>
</td>
<td>
<p><kbd>__host__ __device__</kbd></p>
</td>
<td>
<p><kbd>__host__ __device__</kbd></p>
</td>
<td>
<p>No equivalent</p>
</td>
</tr>
<tr>
<td>
<p><strong>Kernel Launch</strong></p>
</td>
<td>
<p><kbd>&lt;&lt;&lt; &gt;&gt;&gt;</kbd></p>
</td>
<td>
<p><kbd>hipLaunchKernelGGL</kbd></p>
</td>
<td>
<p><kbd>clEnqueueNDRangeKernel</kbd></p>
</td>
</tr>
<tr>
<td>
<p><strong>Term</strong></p>
</td>
<td>
<p>CUDA</p>
</td>
<td>
<p>HIP</p>
</td>
<td>
<p>OpenCL</p>
</td>
</tr>
<tr>
<td>
<p><strong>Global Memory</strong></p>
</td>
<td>
<p><kbd>__global__</kbd></p>
</td>
<td>
<p><kbd>__global__</kbd></p>
</td>
<td>
<p><kbd>__global</kbd></p>
</td>
</tr>
<tr>
<td>
<p><strong>Group Memory</strong></p>
</td>
<td>
<p><kbd>__shared__</kbd></p>
</td>
<td>
<p><kbd>__shared__</kbd></p>
</td>
<td>
<p><kbd>__local</kbd></p>
</td>
</tr>
<tr>
<td>
<p><strong>Constant</strong></p>
</td>
<td>
<p><kbd>__constant__</kbd></p>
</td>
<td>
<p><kbd>__constant__</kbd></p>
</td>
<td>
<p><kbd>__constant</kbd></p>
</td>
</tr>
<tr>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td></td>
<td>
<p><kbd>__syncthreads</kbd></p>
</td>
<td>
<p><kbd>__syncthreads</kbd></p>
</td>
<td>
<p><kbd>barrier(CLK_LOCAL_MEMFENCE)</kbd></p>
</td>
</tr>
<tr>
<td>
<p><strong>Atomic Builtins</strong></p>
</td>
<td>
<p><kbd>atomicAdd</kbd></p>
</td>
<td>
<p><kbd>atomicAdd</kbd></p>
</td>
<td>
<p><kbd>atomic_add</kbd></p>
</td>
</tr>
<tr>
<td>
<p><strong>Term</strong></p>
</td>
<td>
<p>CUDA</p>
</td>
<td>
<p>HIP</p>
</td>
<td>
<p>OpenCL</p>
</td>
</tr>
<tr>
<td>
<p><strong>Precise Math</strong></p>
</td>
<td>
<p><kbd>cos(f)</kbd></p>
</td>
<td>
<p><kbd>cos(f)</kbd></p>
</td>
<td>
<p><kbd>cos(f)</kbd></p>
</td>
</tr>
<tr>
<td>
<p><strong>Fast</strong> <strong>Math</strong></p>
</td>
<td>
<p><kbd>__cos(f)</kbd></p>
</td>
<td>
<p><kbd>__cos(f)</kbd></p>
</td>
<td>
<p><kbd>native_cos(f)</kbd></p>
</td>
</tr>
<tr>
<td>
<p><strong>Vector</strong></p>
</td>
<td>
<p><kbd>float4</kbd></p>
</td>
<td>
<p><kbd>float4</kbd></p>
</td>
<td>
<p><kbd>float4</kbd></p>
</td>
</tr>
</tbody>
</table>

# 安装PyOpenCL for Python（AMD和NVIDIA）